# AI Photography Critique API
**Kaggle Notebook — GPU T4 x2 required**

### Before running:
1. Set **Accelerator → GPU T4 x2** in notebook settings
2. Enable **Internet** in notebook settings
3. Add the following **Kaggle Secrets** (Add-ons → Secrets):
   - `NGROK_AUTH_TOKEN` — from https://dashboard.ngrok.com
   - `HF_TOKEN` — from https://huggingface.co/settings/tokens
4. Create a **public Hugging Face dataset** repo named `api-config` under your username

### Run all cells top to bottom. The server starts in the last cell.

## Cell 1 — Install dependencies

In [1]:
%%capture
!pip uninstall unsloth unsloth-zoo -y
!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git" "unsloth-zoo @ git+https://github.com/unslothai/unsloth-zoo.git" -q
!pip install fastapi uvicorn pyngrok pillow nest_asyncio huggingface_hub python-multipart -q

## Cell 2 — Load secrets

In [2]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = secrets.get_secret("NGROK_AUTH_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

# ── Edit this ──────────────────────────────────────────────────────────────
HF_USERNAME = "dulshanbandara"   # Your Hugging Face username
HF_REPO_ID  = f"{HF_USERNAME}/api-config"  # Must exist as a public dataset repo
MODEL_REPO  = "dulshanbandara/qwen2vl-2b-photography-critique"
# ───────────────────────────────────────────────────────────────────────────

print("Secrets loaded successfully.")

Secrets loaded successfully.


## Cell 3 — Load model

In [3]:

import torch
from unsloth import FastVisionModel

print(f"Loading model: {MODEL_REPO}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_REPO,
    load_in_4bit=True,
    token=HF_TOKEN,
)
FastVisionModel.for_inference(model)

print("Model loaded and ready for inference.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: dulshanbandara/qwen2vl-2b-photography-critique
GPU: Tesla T4
==((====))==  Unsloth 2026.5.2: Fast Qwen2_Vl patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/116M [00:00<?, ?B/s]

Model loaded and ready for inference.


## Cell 4 — Define inference helper

In [4]:
import json
import re
from PIL import Image

SYSTEM_PROMPT = """You are a master photography instructor and aesthetic critic. Evaluate the provided image mechanically and artistically across composition, lighting, and technical execution. Provide scores (1-10) and detailed, actionable critiques for each dimension. Always respond in valid JSON format."""

USER_PROMPT = (
    "Analyse this photograph as an expert photography instructor. "
    "Evaluate composition, lighting, and technical execution. "
    "Provide an overall score, per-dimension scores and critiques, "
    "and a list of actionable improvement steps. "
    "Respond only with a JSON object."
)


def run_critique(image: Image.Image) -> dict:
    """Run the VLM critique on a PIL image and return a parsed dict."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": USER_PROMPT},
            ],
        }
    ]

    # Build the prompt string
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Tokenise — pass image separately for vision models
    inputs = tokenizer(
        images=image,
        text=input_text,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=True,
        )

    # Decode only the newly generated tokens (skip the prompt)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Extract JSON — strip markdown fences if model adds them
    raw_text = re.sub(r"^```(?:json)?\s*", "", raw_text)
    raw_text = re.sub(r"\s*```$", "", raw_text)

    json_start = raw_text.find("{")
    json_end   = raw_text.rfind("}") + 1

    if json_start == -1 or json_end == 0:
        return {"error": "Model did not return valid JSON", "raw": raw_text}

    try:
        return json.loads(raw_text[json_start:json_end])
    except json.JSONDecodeError as e:
        return {"error": f"JSON parse failed: {e}", "raw": raw_text[json_start:json_end]}


print("Inference helper defined.")

Inference helper defined.


## Cell 5 — Define FastAPI app

In [5]:
import io
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI(
    title="AI Photography Critique API",
    description="Vision-Language Model critique endpoint powered by Qwen2-VL-2B",
    version="1.0.0",
)


@app.get("/")
async def root():
    return {
        "status": "live",
        "model": MODEL_REPO,
        "endpoints": {
            "health":  "GET  /health",
            "docs":    "GET  /docs",
            "critique":"POST /critique  (multipart: file=<image>)",
        },
    }


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL_REPO, "gpu": torch.cuda.get_device_name(0)}


@app.post("/critique")
async def critique_photo(file: UploadFile = File(...)):
    # Validate content type
    if file.content_type not in ("image/jpeg", "image/png", "image/webp", "image/jpg"):
        raise HTTPException(
            status_code=415,
            detail=f"Unsupported media type: {file.content_type}. Send JPEG, PNG, or WebP."
        )

    try:
        raw_bytes = await file.read()
        image = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Could not read image: {e}")

    result = run_critique(image)

    if "error" in result:
        raise HTTPException(status_code=500, detail=result)

    return JSONResponse(content=result)


print("FastAPI app defined. Endpoints: GET / | GET /health | POST /critique")

FastAPI app defined. Endpoints: GET / | GET /health | POST /critique


## Cell 6 — Open ngrok tunnel and publish URL to Hugging Face

In [6]:
from pyngrok import ngrok, conf
from huggingface_hub import HfApi

# Kill any existing tunnels from a previous run in this session
ngrok.kill()

# Authenticate and open tunnel
conf.get_default().auth_token = NGROK_AUTH_TOKEN
tunnel = ngrok.connect(8000, bind_tls=True)
PUBLIC_URL = tunnel.public_url
print(f"ngrok tunnel open: {PUBLIC_URL}")

# Publish URL to HF dataset so the mobile app can discover it
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=PUBLIC_URL.encode(),
    path_in_repo="current_api_url.txt",
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    commit_message="Update API URL",
)
print(f"URL published to hf.co/datasets/{HF_REPO_ID}/current_api_url.txt")
print(f"\nInteractive docs: {PUBLIC_URL}/docs")
print(f"Health check:     {PUBLIC_URL}/health")
print(f"Critique endpoint:{PUBLIC_URL}/critique  (POST, multipart image)")

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


ngrok tunnel open: https://unmurmured-incoherently-remi.ngrok-free.dev
URL published to hf.co/datasets/dulshanbandara/api-config/current_api_url.txt

Interactive docs: https://unmurmured-incoherently-remi.ngrok-free.dev/docs
Health check:     https://unmurmured-incoherently-remi.ngrok-free.dev/health
Critique endpoint:https://unmurmured-incoherently-remi.ngrok-free.dev/critique  (POST, multipart image)


## Cell 7 — Start server
This cell runs indefinitely. The API is live while this cell is executing.  
**Interrupt the kernel** (square stop button) to shut down cleanly.

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
server = uvicorn.Server(config)

print("Server starting... (interrupt kernel to stop)")
await server.serve()

Server starting... (interrupt kernel to stop)


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     101.2.177.175:0 - "POST /critique HTTP/1.1" 200 OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     101.2.177.175:0 - "POST /critique HTTP/1.1" 200 OK


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     101.2.177.175:0 - "POST /critique HTTP/1.1" 200 OK


## Cell 8 — Quick local test (optional, run in a separate kernel tab)
Run this in a *new* notebook tab or after interrupting the server cell.

In [ ]:
import requests

# ── Replace with your actual ngrok URL ────────────────────────────────────
BASE_URL = https://unmurmured-incoherently-remi.ngrok-free.dev   # or paste the URL string directly as a fallback
TEST_IMAGE_PATH = "/kaggle/input/your-dataset/sample.jpg"  # adjust path
# ──────────────────────────────────────────────────────────────────────────

# Health check
r = requests.get(f"{BASE_URL}/health", timeout=10)
print("Health:", r.json())

# Critique
with open(TEST_IMAGE_PATH, "rb") as f:
    r = requests.post(
        f"{BASE_URL}/critique",
        files={"file": ("photo.jpg", f, "image/jpeg")},
        timeout=60,
    )

print("Status:", r.status_code)
print(json.dumps(r.json(), indent=2))